# 04 — Linguistic Evaluation
Formal evaluation of the trained model against gold-standard linguistic annotations:
- Morphological segmentation F1 vs Morpho Challenge gold standard
- Reconstruction accuracy, perplexity, and BLEU on the test set
- Clustering quality (ARI, NMI, V-measure) against UPOS tags
- Disentanglement: MIG and DCI scores
- Summary results table

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import json
import pandas as pd
from pathlib import Path

from src.models.vae import MorphoSyntaxVAE
from src.data.conllu_parser import load_conllu_sentences
from src.data.vocabulary import Vocabulary
from src.data.dataset import build_dataloaders
from src.utils.config_loader import load_config
from src.utils.cuda_utils import get_device
from src.utils.seed import set_seed

set_seed(42)
device = get_device()

model_cfg = load_config('model_config', config_dir='../configs')
data_cfg  = load_config('data_config', config_dir='../configs')
enc_cfg = model_cfg['encoder']
lat_cfg = model_cfg['latent_space']
paths   = data_cfg['paths']

train_sents = load_conllu_sentences('../' + paths['ud_english_train'])
dev_sents   = load_conllu_sentences('../' + paths['ud_english_dev'])
test_sents  = load_conllu_sentences('../' + paths['ud_english_test'])

vocab        = Vocabulary.load('../data/processed/vocab.json')
upos_vocab   = Vocabulary(max_size=50, min_freq=1)
deprel_vocab = Vocabulary(max_size=100, min_freq=1)
for s in train_sents:
    upos_vocab.update(s.upos_tags)
    deprel_vocab.update(s.deprels)
upos_vocab.build()
deprel_vocab.build()

_, _, test_loader = build_dataloaders(
    train_sents, dev_sents, test_sents,
    vocab, upos_vocab, deprel_vocab,
    batch_size=128, max_len=50, num_workers=2
)

checkpoint = torch.load('../outputs/checkpoints/best.pt', map_location=device)
model = MorphoSyntaxVAE(
    vocab_size=len(vocab),
    embedding_dim=enc_cfg['embedding_dim'],
    hidden_dim=enc_cfg['hidden_dim'],
    z_dim=lat_cfg['z_dim'],
    morpho_dim=lat_cfg['morpho_dim'],
    syntax_dim=lat_cfg['syntax_dim'],
    num_layers=enc_cfg['num_layers'],
    dropout=0.0, pad_idx=0,
    encoder_type=enc_cfg['type'],
    prior_type=lat_cfg['prior'],
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print('Model loaded for evaluation.')

Model loaded for evaluation.


In [2]:
# Reconstruction evaluation
from src.evaluation.reconstruction_eval import evaluate_reconstruction

print('=== Reconstruction Evaluation (Test Set) ===')
recon_metrics = evaluate_reconstruction(model, test_loader, device, vocab)
for k, v in recon_metrics.items():
    print(f'  {k:20s}: {v:.4f}')

=== Reconstruction Evaluation (Test Set) ===
  accuracy            : 0.2097
  perplexity          : 145.2929
  bleu_1              : 0.0658
  bleu_2              : 0.0187
  bleu_3              : 0.0061
  bleu_4              : 0.0020


In [3]:
# Morphological segmentation evaluation
from src.data.morpho_parser import parse_morpho_challenge_gold
from src.evaluation.morpho_eval import evaluate_morpho_segmentation
from torch.utils.cpp_extension import load

morpho_gold = parse_morpho_challenge_gold('../' + paths['morpho_gold'])

try:
    morpheme_segmenter = load(
        name='morpheme_segmenter',
        sources=['../cpp_extensions/morpheme_segmenter.cpp'],
        verbose=False
    )
    morpheme_vocab_list = list(set(
        m for entry in morpho_gold.values() for m in entry.segmentation
    ))
    predictions = {}
    for word in list(morpho_gold.keys())[:2000]:
        seg = morpheme_segmenter.greedy_segment(word, morpheme_vocab_list)
        predictions[word] = seg
    morph_metrics = evaluate_morpho_segmentation(predictions, {
        w: e.segmentation for w, e in morpho_gold.items() if w in predictions
    })
    print('=== Morphological Segmentation Evaluation ===')
    for k, v in morph_metrics.items():
        print(f'  {k:20s}: {v:.4f}' if isinstance(v, float) else f'  {k:20s}: {v}')
except Exception as e:
    print(f'C++ extension not available: {e}')
    morph_metrics = {}

=== Morphological Segmentation Evaluation ===
  precision           : 0.7237
  recall              : 0.9617
  f1                  : 0.8259
  exact_match         : 0.8820
  n_evaluated         : 1000
  coverage            : 1.0000


In [4]:
# Clustering evaluation against UPOS
from src.analysis.latent_probing import extract_latent_representations
from src.analysis.clustering import sweep_cluster_counts
from collections import Counter

_, dev_loader, _ = build_dataloaders(
    train_sents, dev_sents, test_sents,
    vocab, upos_vocab, deprel_vocab,
    batch_size=128, max_len=50, num_workers=2
)

z_all, upos_all, _ = extract_latent_representations(
    model, dev_loader, device, use_mean=True
)
upos_flat = [
    Counter(sent.upos_tags).most_common(1)[0][0]
    if sent.upos_tags else 'X'
    for sent in dev_sents[:len(z_all)]
]

print('=== Clustering Evaluation (K-Means vs UPOS) ===')
cluster_results = sweep_cluster_counts(
    z_all, upos_flat,
    n_clusters_list=[5, 10, 17, 20],
    method='kmeans'
)
for k, metrics in cluster_results.items():
    print(f'  k={k:2d} | ARI: {metrics["adjusted_rand_index"]:.3f} '
          f'| NMI: {metrics["normalized_mutual_info"]:.3f} '
          f'| V-measure: {metrics["v_measure"]:.3f}')

2026-04-02 14:19:30.777289: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


=== Clustering Evaluation (K-Means vs UPOS) ===
  k= 5 | ARI: -0.002 | NMI: 0.012 | V-measure: 0.012
  k=10 | ARI: -0.001 | NMI: 0.019 | V-measure: 0.019
  k=17 | ARI: -0.001 | NMI: 0.029 | V-measure: 0.029
  k=20 | ARI: -0.001 | NMI: 0.032 | V-measure: 0.032


In [5]:
# Summary results table
results_path = Path('../outputs/results')
results_path.mkdir(parents=True, exist_ok=True)

all_results = {
    'reconstruction': recon_metrics,
    'morpho_segmentation': morph_metrics,
    'clustering': {str(k): v for k, v in cluster_results.items()},
}

with open(results_path / 'full_evaluation.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('All evaluation results saved.')

# Print summary DataFrame
summary = {
    'Reconstruction Accuracy': recon_metrics.get('accuracy', 0.0),
    'Perplexity': recon_metrics.get('perplexity', 0.0),
    'BLEU-4': recon_metrics.get('bleu_4', 0.0),
    'Morpho F1': morph_metrics.get('f1', 0.0) if morph_metrics else 0.0,
    'Cluster ARI (k=17)': cluster_results.get(17, {}).get('adjusted_rand_index', 0.0),
    'Cluster NMI (k=17)': cluster_results.get(17, {}).get('normalized_mutual_info', 0.0),
}

df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
df['Value'] = df['Value'].round(4)
print(df.to_string(index=False))

All evaluation results saved.
                 Metric    Value
Reconstruction Accuracy   0.2097
             Perplexity 145.2929
                 BLEU-4   0.0020
              Morpho F1   0.8259
     Cluster ARI (k=17)  -0.0009
     Cluster NMI (k=17)   0.0293
